In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [2]:
import random

In [3]:
random.seed(42)

In [4]:
tests = ['ks_30_0', 'ks_50_0', 'ks_200_0', 'ks_400_0', 'ks_1000_0', 'ks_10000_0']
thresholds = [(92_000, 99_798), (141_956, 142_156), (100_062, 100_236), (3_966_813, 3_967_028), 
              (109_869, 109_899), (1_099_870, 1_099_881)]

In [5]:
def load_knapsack(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, W = list(map(int, lines[0].split()))
        cost = list()
        weight = list()
        for i in range(n):
            v, w = list(map(int, lines[1 + i].split()))
            cost.append(v)
            weight.append(w)

        return n, W, cost, weight

In [6]:
def check_knapsack(n, W, cost, weight, chosen):
    sm = 0
    result = 0
    for i in chosen: 
        sm += weight[i]
        result += cost[i]
        
    if sm > W:
        raise Exception("Total weight exceeds W")
         
    return result

In [7]:
def passed_cnt(result, idx):
    if result >= thresholds[idx][1]:
        return 2
    elif result >= thresholds[idx][0]:
        return 1
    else:
        return 0

In [8]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, W, cost, weight = load_knapsack(test)
        start = time.time()
        
        if not use_file:
            chosen = method(n, W, cost, weight)
        else:
            chosen = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_knapsack(n, W, cost, weight, chosen)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Сначала напишем стандартную динамику для рюкзака $dp[n][W]$ = максимальная стоимость, которую можно получить используя первые $n$ объектов, что их сумма весов = $W$. Также заметим, что данный метод дает оптимум на тест.

In [9]:
!g++ -std=c++17 cpp_methods/dp.cpp -o tmp/dp

In [12]:
def dp_knapsack(test_file): 
    os.system(f"./tmp/dp data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        chosen = list(map(int, lines[0].split()))
        return chosen

In [13]:
test_method(dp_knapsack, "dp", True)

Checking dp
Execution time: 0.0811 seconds
Target function ks_30_0: 99798
Passed ks_30_0: 2
Execution time: 0.2606 seconds
Target function ks_50_0: 142156
Passed ks_50_0: 2
Execution time: 0.2506 seconds
Target function ks_200_0: 100236
Passed ks_200_0: 2
Execution time: 61.6412 seconds
Target function ks_400_0: 3967180
Passed ks_400_0: 2
Execution time: 1.4273 seconds
Target function ks_1000_0: 109899
Passed ks_1000_0: 2
Execution time: 141.8325 seconds
Target function ks_10000_0: 1099893
Passed ks_10000_0: 2
Score: 30


Получили оптимумы на все тесты, но долго работают тесты: `ks_400_0`, `ks_10000_0`, что довольно логично, так как время работы нашего алгоритма $\mathcal{O}(nW)$.

Попробуем теперь воспользоваться жадным алгоритмом, который пытается взять предметы в рюкзак в порядке убывания удельной стоимости $\left( \frac{v_i}{w_i} \right)$.

In [14]:
!g++ -std=c++17 cpp_methods/greedy.cpp -o tmp/greedy

In [15]:
def greedy_knapsack(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        chosen = list(map(int, lines[0].split()))
        return chosen

In [16]:
test_method(greedy_knapsack, "greedy", True)

Checking greedy
Execution time: 0.4337 seconds
Target function ks_30_0: 90000
Passed ks_30_0: 0
Execution time: 0.0169 seconds
Target function ks_50_0: 141956
Passed ks_50_0: 1
Execution time: 0.0129 seconds
Target function ks_200_0: 100062
Passed ks_200_0: 1
Execution time: 0.0091 seconds
Target function ks_400_0: 3966813
Passed ks_400_0: 1
Execution time: 0.0087 seconds
Target function ks_1000_0: 109869
Passed ks_1000_0: 1
Execution time: 0.0273 seconds
Target function ks_10000_0: 1099870
Passed ks_10000_0: 1
Score: 15


Проходит почти все первые пороги, кроме первого теста. Также стоит отметить, что время работы алгоритма очень маленькое.

Не забудем, что у нас есть теория, где есть небольшая модификация жадного алгоритма: 
Пусть $k + 1$ - это первый не взятый жадным алгоритмом элемент в правильном порядке. Тогда предлагается расмотреть лучший из ответов:

1. Ответ, полученный первым алгоритмом.
2. Ответ, полученный первым алгоритмом, но пропускающий первые $k$ элементов.

Данный метод даст $\frac{1}{2}$ аппроксимацию, но мы его немного модифицируем. Мы переберем все возможные пропуски первых элементов, таким образом получим решение за $\mathcal{O}(n^2)$.

In [17]:
!g++ -std=c++17 cpp_methods/tuned_greedy.cpp -o tmp/tuned_greedy

In [18]:
def tuned_greedy_knapsack(test_file): 
    os.system(f"./tmp/tuned_greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        chosen = list(map(int, lines[0].split()))
        return chosen

In [19]:
test_method(tuned_greedy_knapsack, "tuned_greedy", True)

Checking tuned_greedy
Execution time: 0.1963 seconds
Target function ks_30_0: 99751
Passed ks_30_0: 1
Execution time: 0.0115 seconds
Target function ks_50_0: 141956
Passed ks_50_0: 1
Execution time: 0.0112 seconds
Target function ks_200_0: 100236
Passed ks_200_0: 2
Execution time: 0.0144 seconds
Target function ks_400_0: 3966813
Passed ks_400_0: 1
Execution time: 0.0132 seconds
Target function ks_1000_0: 109869
Passed ks_1000_0: 1
Execution time: 0.1435 seconds
Target function ks_10000_0: 1099870
Passed ks_10000_0: 1
Score: 20


Модификация жадника дала небольшое улучшение в результатах!

Заметим, `upper-bound` на ответ - это ответ, полученных задачей дробного рюкзака. Воспользуемся этим и напишем рекурсивный алгоритм, который пытается набирать объекты, следующим образом:

1. Берем исходно жадное приближение.
2. Сортируем объекты по удельной стоимости (cost/weight).
3. Делаем рекурсию по берем / не берем элемент и храним лучший текущий ответ.
4. Если `upper-bound` ветки <= текущего оптимума выходим из рекурсии.
5. Достраиваем жадно решение (первым методом) и возможно обновляем оптимум.

In [20]:
!g++ -std=c++17 cpp_methods/recursive.cpp -o tmp/recursive

In [21]:
def rec_knapsack(test_file): 
    os.system(f"./tmp/recursive data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        chosen = list(map(int, lines[0].split()))
        return chosen

In [22]:
test_method(rec_knapsack, "recursive", True)

Checking recursive
Execution time: 0.2096 seconds
Target function ks_30_0: 99798
Passed ks_30_0: 2
Execution time: 0.0108 seconds
Target function ks_50_0: 142156
Passed ks_50_0: 2
Execution time: 8.5389 seconds
Target function ks_200_0: 100236
Passed ks_200_0: 2
Execution time: 0.1020 seconds
Target function ks_400_0: 3967180
Passed ks_400_0: 2
Execution time: 0.0354 seconds
Target function ks_1000_0: 109899
Passed ks_1000_0: 2
Execution time: 3.3028 seconds
Target function ks_10000_0: 1099893
Passed ks_10000_0: 2
Score: 30


Мы прошли все пороги и максимальное время стало всего лишь ~ 8 секунд, ура! Данный алгоритм выступит нашим основным решением.

Заметим, что наш алгоритм не делает никогда некорректных отсечений, поэтому ответ, полученный им также оптимален.

Итоги текущих улучшений:
1. Динамическое программирование показало себя хорошо в заданных ограничениях и почти везде отработало быстро, но было пару тестов, где оно все еще медленно. Однако данный метод дает всегда оптимум.
2. Жадный алгоритм через удельные стоимости почти прошло все простые пороги.
3. Небольшая модификация (перебор старта) прошла уже все простые пороги и один сложный порог.
4. Реализовал идею похожую на ту, что была на семинаре. Где мы делаем рекурсию с отсечениями по верхним границам и считаем каждый раз новую нижнюю границу. Метод сработал чрезвычайно быстро на всех тестах, хотя он тоже дает всегда оптимальный ответ.